# 🌍 Global News Data Project: Beginner & Student Guide
## Phase 1: Collecting, Cleaning & Preparing GDELT News Data

**Author / Responsible:** Shahida  
**Project Teammates:** Suraiya & Jakia   
**Project Title:** Mapping the Geography of News Coverage  
**Date Range We Are Studying:** July 14, 2026 to October 13, 2026 (Exactly 3 months, inclusive)  
**Geographic Scope:** Worldwide! (Africa, Asia, Europe, North America, South America, and Oceania)

---

### 👋 Welcome! What is this notebook about?
Imagine you want to create a world map showing what the news talks about every day. Which countries get the most attention? What topics are trending? Are some parts of the world rarely mentioned?

To answer these questions, we need **real, high-quality news data**. In this notebook, we walk through the entire process step-by-step:
1. **Source Exploration:** We test the GDELT DOC 2.0 API, observe its rate limits, and transition to GDELT's open 15-minute data stream.
2. **Data Collection:** We collect real news records across the entire 3-month period without making up fake data.
3. **Missing Value Inspection:** We audit missing values across all source fields.
4. **Duplicate Removal:** We remove duplicate URLs and syndicated news copies.
5. **Country & Continent Mapping & Filtering:** We resolve locations mentioned in the article text to official ISO country codes and continents. Since our goal is **Mapping the Geography of News Coverage**, we filter out articles without any identifiable country so that **100% of articles in our final dataset have a verified country and continent**!
6. **Smart Topic Categorization:** We use an improved classification method that combines article headlines and theme tags to sort articles into 11 clean categories (*Health*, *Technology*, *Politics*, etc.) while preventing false-positive keyword mistakes (like confusing *warehousing* with *war*).
7. **Publisher Cleaning:** We clean website domain names into standardized publisher names (like `bbc.com`).
8. **Automated Quality Checks:** We run automated tests verifying dates, country codes, continents, topics, and URLs.
9. **Saving the Deliverables:** We save the final dataset as `gdelt_clean.csv` and document every column in `data_dictionary.csv`.

Let's begin! 🚀


In [1]:
# ==============================================================================
# Step 0: Import the tools (Python libraries) we need
# ==============================================================================

# 'sys' and 'os' help Python interact with files and system settings
import sys
import os

# 'time' measures execution speed
import time

# 'csv', 'io', 'zipfile', 'json', 're', 'html', and 'urllib' handle text, web, and archive files
import csv
import io
import json
import zipfile
import urllib.parse
import html
import re

# 'requests' downloads files from the web
import requests

# 'pandas' manages our data in organized tables (DataFrames)
import pandas as pd

# 'pycountry' provides the official international database of country names and ISO codes
import pycountry

# 'pycountry_convert' maps country codes to their respective continents
import pycountry_convert as pc

# In GDELT, text fields can be large. We increase Python's CSV field size limit
# so it never crashes when parsing long articles.
csv.field_size_limit(sys.maxsize)

print("✅ All libraries loaded successfully!")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")


✅ All libraries loaded successfully!
Python version: 3.13.9
Pandas version: 2.3.3


## Step 1: Testing the GDELT DOC 2.0 API (Our First Candidate)

### What is GDELT?
**GDELT** stands for the *Global Database of Events, Language, and Tone*. It monitors broadcast, print, and web news across the globe in over 65 languages.

### Why did we test GDELT DOC 2.0 API first?
The project instructions suggest trying the **GDELT DOC 2.0 API** first (`https://api.gdeltproject.org/api/v2/doc/doc`).

### What did we find? (The Operational Limits):
1. **Strict Rate Limiting:** The API enforces a limit of **1 request every 5 seconds**. Fast queries trigger `HTTP 429: Too Many Requests`.
2. **250-Record Ceiling:** Each query is capped at a maximum of **250 articles** without cursor or offset pagination.
3. **The Big Trap — "Publisher Country" vs "Article Country":**
   - The DOC 2.0 API returns `sourcecountry`. This is where the **newspaper server** is based, **not** what the story is about!
   - *Example:* If Reuters in the UK writes about an earthquake in Japan, `sourcecountry` says United Kingdom. If we mapped that, our map would incorrectly show UK news instead of Japanese news!
   - Additionally, DOC 2.0 does not provide structured topic categories in article search mode.

### 💡 The Solution: GDELT 2.0 Global Knowledge Graph (GKG)
GDELT publishes its raw knowledge graph files every 15 minutes:
- **No rate limits** and no 250-record cap.
- **`V1Locations`:** Identifies the exact cities and countries mentioned *inside the article text*.
- **`V1Themes`:** Provides detailed thematic codes.
- **`<PAGE_TITLE>`:** Captures the full headline.

Let's test the DOC 2.0 API to see how it responds.


In [2]:
# ==============================================================================
# Testing the GDELT DOC 2.0 API to observe its behavior
# ==============================================================================

api_url = "https://api.gdeltproject.org/api/v2/doc/doc"

query_parameters = {
    "query": "international news",
    "mode": "ArtList",
    "maxrecords": "5",
    "format": "json"
}

headers = {"User-Agent": "StudentResearchProject/1.0"}

try:
    response = requests.get(api_url, params=query_parameters, headers=headers, timeout=10)
    print(f"API Response Code: {response.status_code}")
    
    if response.status_code == 200:
        data = response.json()
        articles = data.get("articles", [])
        print(f"Successfully retrieved {len(articles)} sample article(s).")
        if articles:
            print("Fields returned by DOC 2.0:")
            for field_name in articles[0].keys():
                print(f"  - {field_name}")
    elif response.status_code == 429:
        print("Notice: Received HTTP 429 (Rate Limit). This confirms why the API has strict throttling!")
except Exception as err:
    print(f"API test encountered: {err}")


API Response Code: 429
Notice: Received HTTP 429 (Rate Limit). This confirms why the API has strict throttling!


## Step 2: Collecting Real Worldwide News Across 3 Months

### Date Range:
- **Start Date:** July 14, 2026
- **End Date:** October 13, 2026
- **Total Duration:** Exactly 3 months (92 days, inclusive)

### Sampling Strategy:
To collect a balanced, worldwide sample across the entire period, we sample **19 regular time slices** spaced about every 4 to 5 days:
- July: July 14, 19, 24, 29
- August: August 3, 8, 13, 18, 23, 28
- September: September 2, 7, 12, 17, 22, 27
- October: October 2, 7, 13

We stream each 15-minute archive into memory and extract:
- `article_id`: Unique GDELT identifier.
- `publication_date`: Timestamp when the article was published.
- `title`: Clean headline from `<PAGE_TITLE>`.
- `publisher`: News source domain.
- `article_url`: Original web link.
- `raw_locations`: Countries and places mentioned in the text.
- `raw_themes`: Thematic tags recorded by GDELT.


In [3]:
# ==============================================================================
# Step 2: Download news archives across our 19 sampling dates
# ==============================================================================

# Timestamps in format: YYYYMMDDHHMMSS (120000 = 12:00 PM noon UTC)
SAMPLE_TIMESTAMPS = [
    "20260714120000", "20260719120000", "20260724120000", "20260729120000",
    "20260803120000", "20260808120000", "20260813120000", "20260818120000",
    "20260823120000", "20260828120000", "20260902120000", "20260907120000",
    "20260912120000", "20260917120000", "20260922120000", "20260927120000",
    "20261002120000", "20261007120000", "20261013120000"
]

print(f"Starting data collection across {len(SAMPLE_TIMESTAMPS)} dates...")
start_time = time.time()

collected_articles = []

for timestamp in SAMPLE_TIMESTAMPS:
    file_url = f"https://data.gdeltproject.org/gdeltv2/{timestamp}.gkg.csv.zip"
    
    try:
        response = requests.get(file_url, timeout=30)
        
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
                inner_csv_name = zip_ref.namelist()[0]
                with zip_ref.open(inner_csv_name) as file_handle:
                    reader = csv.reader(
                        io.TextIOWrapper(file_handle, encoding="utf-8", errors="ignore"),
                        delimiter="\t"
                    )
                    
                    articles_in_batch = 0
                    for row in reader:
                        if not row or len(row) < 27:
                            continue
                        
                        record_id = row[0].strip()       # Col 0: GKG Record ID
                        date_raw = row[1].strip()        # Col 1: Date (YYYYMMDDHHMMSS)
                        publisher_raw = row[3].strip()   # Col 3: Publisher domain
                        url_raw = row[4].strip()         # Col 4: Article URL
                        themes_raw = row[7].strip()      # Col 7: Themes
                        locations_raw = row[9].strip()   # Col 9: Locations in text
                        xml_raw = row[26]                # Col 26: Extras XML
                        
                        # Extract the headline from <PAGE_TITLE>
                        article_title = ""
                        title_match = re.search(r"<PAGE_TITLE>(.*?)</PAGE_TITLE>", xml_raw, re.DOTALL)
                        if title_match:
                            article_title = html.unescape(title_match.group(1)).strip()
                        
                        collected_articles.append({
                            "article_id": record_id,
                            "raw_date": date_raw,
                            "raw_publisher": publisher_raw,
                            "article_url": url_raw,
                            "title": article_title,
                            "raw_themes": themes_raw,
                            "raw_locations": locations_raw
                        })
                        articles_in_batch += 1
                    
                    readable_date = f"{timestamp[:4]}-{timestamp[4:6]}-{timestamp[6:8]}"
                    print(f"  ✓ {readable_date}: Loaded {articles_in_batch:,} articles")
        else:
            print(f"  ✗ Could not fetch {timestamp} (HTTP {response.status_code})")
    except Exception as error:
        print(f"  ✗ Error on {timestamp}: {error}")

df_raw = pd.DataFrame(collected_articles)
total_downloaded = len(df_raw)
elapsed = time.time() - start_time
print(f"\n🎉 Finished! Collected {total_downloaded:,} raw articles in {elapsed:.2f} seconds.")


Starting data collection across 19 dates...


  ✓ 2026-07-14: Loaded 1,200 articles


  ✓ 2026-07-19: Loaded 605 articles


  ✓ 2026-07-24: Loaded 1,076 articles


  ✓ 2026-07-29: Loaded 1,260 articles


...

  ✓ 2026-09-22: Loaded 1,323 articles


  ✓ 2026-09-27: Loaded 670 articles


  ✓ 2026-10-02: Loaded 1,220 articles


  ✓ 2026-10-07: Loaded 1,226 articles


  ✓ 2026-10-13: Loaded 1,118 articles

🎉 Finished! Collected 19,406 raw articles in 7.01 seconds.


## Step 3: Inspecting Raw Fields & Missing Values

Let's inspect the raw dataset structure and check for missing values across attributes.

### 💡 Understanding Missingness in News Data:
- In real news monitoring, not every article mentions a specific country. For example, generic lifestyle tips, product reviews, or philosophical essays often mention zero geographic places.
- We never invent or fabricate fake countries for articles that lack them.


In [4]:
# ==============================================================================
# Step 3: Inspect the raw table and missing values
# ==============================================================================

print("--- First 5 rows of our dataset ---")
display(df_raw[["article_id", "raw_date", "raw_publisher", "title", "article_url"]].head())

print("\n--- Missing Values in Raw Fields ---")
missing_count = df_raw.isnull().sum()
missing_percent = (missing_count / len(df_raw)) * 100

audit_table = pd.DataFrame({
    "Missing Values Count": missing_count,
    "Percentage Missing (%)": missing_percent.round(2)
})
display(audit_table)


--- First 5 rows of our dataset ---


,article_id,raw_date,raw_publisher,title,article_url
0,20260714120000-0,20260714120000,irrigator.com.au,Man accused of assigning attacks on venues fac...,https://www.irrigator.com.au/story/9310322/man...
1,20260714120000-1,20260714120000,vanguardngr.com,"US strikes Iran, vows to reimpose naval blockade",https://www.vanguardngr.com/2026/07/us-strikes...
2,20260714120000-2,20260714120000,queanbeyanage.com.au,'ISIS bride' allegedly urged kids to be like t...,https://www.queanbeyanage.com.au/story/9310787...
3,20260714120000-3,20260714120000,countytimes.co.uk,RNLI Barmouth called out several times on busy...,https://www.countytimes.co.uk/news/26278293.rn...
4,20260714120000-4,20260714120000,walesonline.co.uk,Not SPF — check a number on your sunscreen thi...,https://www.walesonline.co.uk/news/uk-news/thr...



--- Missing Values in Raw Fields ---


,Missing Values Count,Percentage Missing (%)
article_id,0,0.0
raw_date,0,0.0
raw_publisher,0,0.0
article_url,0,0.0
title,0,0.0
raw_themes,0,0.0
raw_locations,0,0.0


## Step 4: Finding and Removing Duplicates

We identify and remove duplicate records:
1. **Duplicate URLs:** Removing repeated indexing of the exact same web page.
2. **Syndicated Near-Duplicates:** Removing wire service copies with identical titles, publishers, and publication dates.
3. **Date Verification:** Confirming every article falls strictly between July 14, 2026 and October 13, 2026.


In [5]:
# ==============================================================================
# Step 4: Detect and remove duplicate records
# ==============================================================================

initial_count = len(df_raw)
print(f"Starting with: {initial_count:,} records")

# 1. Exact duplicates
exact_duplicates = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")

# 2. Duplicate URLs
duplicate_urls = df_raw.duplicated(subset=["article_url"]).sum()
print(f"Duplicate article URLs: {duplicate_urls}")

# Deduplicate by URL (keeping the first occurrence)
df_clean_step = df_raw.drop_duplicates(subset=["article_url"], keep="first").copy()
after_url_dedup = len(df_clean_step)
print(f"Records after URL deduplication: {after_url_dedup:,}")

# 3. Format date to YYYY-MM-DD
df_clean_step["publication_date"] = df_clean_step["raw_date"].apply(
    lambda d: f"{str(d)[0:4]}-{str(d)[4:6]}-{str(d)[6:8]}" if len(str(d)) >= 8 else None
)

# 4. Remove syndicated duplicates (same title + publisher + date)
meaningful_titles = df_clean_step[df_clean_step["title"].str.len() > 15]
title_duplicates = meaningful_titles.duplicated(subset=["title", "raw_publisher", "publication_date"], keep="first")
duplicate_indices = meaningful_titles[title_duplicates].index

df_clean_step = df_clean_step.drop(index=duplicate_indices).copy()
after_title_dedup = len(df_clean_step)
print(f"Records after removing duplicate (title, publisher, date): {after_title_dedup:,}")
print(f"Total duplicates removed: {initial_count - after_title_dedup:,}")

# 5. Filter strictly to requested date window: July 14, 2026 – October 13, 2026
df_clean_step = df_clean_step[
    (df_clean_step["publication_date"] >= "2026-07-14") & 
    (df_clean_step["publication_date"] <= "2026-10-13")
].copy()
print(f"Records confirmed inside the 3-month window: {len(df_clean_step):,}")


Starting with: 19,406 records
Exact duplicate rows: 0
Duplicate article URLs: 0
Records after URL deduplication: 19,406
Records after removing duplicate (title, publisher, date): 18,970
Total duplicates removed: 436
Records confirmed inside the 3-month window: 18,970


## Step 5: Country & Continent Mapping + Geographic Filtering

### How do we map countries and continents?
1. **Content Geography:** We extract countries from `raw_locations` (`V1Locations`), capturing what the news is *talking about*, not where the publisher is located.
2. **ISO 3166-1 alpha-2 Codes:** We convert place names into official 2-letter country codes (e.g., `US`, `GB`, `AU`, `IN`, `NG`, `JP`, `BR`).
3. **Continents:** We assign each country to its continent (**Africa**, **Asia**, **Europe**, **North America**, **South America**, **Oceania**).
4. **Preserving Multi-Country Stories:** If an article mentions multiple countries, the primary country goes into `country`, and all mentioned countries are preserved in `all_countries`, `all_country_codes`, and `all_continents` without duplicating rows.

### 🗺️ Geographic Filtering for Our Mapping Project:
Since the core objective of this project is **Mapping the Geography of News Coverage**, articles that do not mention any identifiable country cannot be mapped on a world map. 

Therefore, we **filter out articles where country is missing**, ensuring that **100% of the articles in our final dataset have a verified country, country code, and continent**!


In [6]:
# ==============================================================================
# Step 5: Country and Continent Resolution + Geographic Filtering
# ==============================================================================

# Build country name to ISO 2-letter code dictionary
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name.lower()] = country.alpha_2
    if hasattr(country, "common_name"):
        country_name_to_iso[country.common_name.lower()] = country.alpha_2

common_aliases = {
    "united states": "US", "usa": "US", "u.s.": "US", "united states of america": "US",
    "united kingdom": "GB", "uk": "GB", "u.k.": "GB", "great britain": "GB", "britain": "GB",
    "russia": "RU", "russian federation": "RU",
    "south korea": "KR", "korea, republic of": "KR",
    "north korea": "KP", "korea, democratic people's republic of": "KP",
    "syria": "SY", "iran": "IR", "iran, islamic republic of": "IR",
    "taiwan": "TW", "vietnam": "VN", "laos": "LA", "bolivia": "BO", "venezuela": "VE",
    "tanzania": "TZ", "dr congo": "CD", "democratic republic of the congo": "CD",
    "congo": "CG", "cote d'ivoire": "CI", "ivory coast": "CI", "czech republic": "CZ",
    "czechia": "CZ", "burma": "MM", "myanmar": "MM", "palestine": "PS", "vatican": "VA",
    "turkey": "TR", "turkiye": "TR"
}
country_name_to_iso.update(common_aliases)

# FIPS 10-4 to ISO 3166-1 alpha-2 Crosswalk
fips_to_iso_crosswalk = {
    "AF": "AF", "AL": "AL", "AG": "DZ", "AN": "AD", "AO": "AO", "AC": "AG", "AR": "AR", "AM": "AM",
    "AS": "AU", "AU": "AT", "AJ": "AZ", "BF": "BS", "BA": "BH", "BG": "BD", "BB": "BB", "BO": "BY",
    "BE": "BE", "BH": "BZ", "BN": "BJ", "BT": "BT", "BL": "BO", "BK": "BA", "BC": "BW", "BR": "BR",
    "BX": "BN", "BU": "BG", "UV": "BF", "BY": "BI", "CB": "KH", "CM": "CM", "CA": "CA", "CV": "CV",
    "CT": "CF", "CD": "TD", "CI": "CL", "CH": "CN", "CO": "CO", "CN": "KM", "CF": "CG", "CG": "CD",
    "CS": "CR", "IV": "CI", "HR": "HR", "CU": "CU", "CY": "CY", "EZ": "CZ", "DA": "DK", "DJ": "DJ",
    "DO": "DM", "DR": "DO", "EC": "EC", "EG": "EG", "ES": "SV", "EK": "GQ", "ER": "ER", "EN": "EE",
    "WZ": "SZ", "ET": "ET", "FJ": "FJ", "FI": "FI", "FR": "FR", "GB": "GA", "GA": "GM", "GG": "GE",
    "GM": "DE", "GH": "GH", "GR": "GR", "GJ": "GD", "GT": "GT", "GV": "GN", "PU": "GW", "GY": "GY",
    "HA": "HT", "HO": "HN", "HU": "HU", "IC": "IS", "IN": "IN", "ID": "ID", "IR": "IR", "IZ": "IQ",
    "EI": "IE", "IS": "IL", "IT": "IT", "JM": "JM", "JA": "JP", "JO": "JO", "KZ": "KZ", "KE": "KE",
    "KR": "KI", "KN": "KP", "KS": "KR", "KU": "KW", "KG": "KG", "LA": "LA", "LG": "LV", "LE": "LB",
    "LT": "LS", "LI": "LR", "LY": "LY", "LS": "LI", "LH": "LT", "LU": "LU", "MC": "MO", "MK": "MK",
    "MA": "MG", "MI": "MW", "MY": "MY", "MV": "MV", "ML": "ML", "MT": "MT", "MR": "MR", "MP": "MU",
    "MX": "MX", "FM": "FM", "MD": "MD", "MN": "MC", "MG": "MN", "MJ": "ME", "MO": "MA", "MZ": "MZ",
    "WA": "NA", "NR": "NR", "NP": "NP", "NL": "NL", "NZ": "NZ", "NU": "NI", "NG": "NE", "NI": "NG",
    "NO": "NO", "MU": "OM", "PK": "PK", "PS": "PW", "PM": "PA", "PP": "PG", "PA": "PY", "PE": "PE",
    "RP": "PH", "PL": "PL", "PO": "PT", "QA": "QA", "RO": "RO", "RS": "RU", "RW": "RW", "SC": "KN",
    "ST": "LC", "VC": "VC", "WS": "WS", "SM": "SM", "TP": "ST", "SA": "SA", "SG": "SN", "RI": "RS",
    "SE": "SC", "SL": "SL", "SN": "SG", "LO": "SK", "SI": "SI", "BP": "SB", "SO": "SO", "SF": "ZA",
    "SS": "SS", "SP": "ES", "CE": "LK", "SU": "SD", "NS": "SR", "SW": "SE", "SZ": "CH", "SY": "SY",
    "TW": "TW", "TI": "TJ", "TZ": "TZ", "TH": "TH", "TO": "TG", "TN": "TO", "TD": "TT", "TS": "TN",
    "TU": "TR", "TX": "TM", "TV": "TV", "UG": "UG", "UP": "UA", "AE": "AE", "UK": "GB", "US": "US",
    "UY": "UY", "UZ": "UZ", "NH": "VU", "VE": "VE", "VM": "VN", "YM": "YE", "ZA": "ZM", "ZI": "ZW"
}

def get_continent_for_country(iso_code):
    try:
        continent_code = pc.country_alpha2_to_continent_code(iso_code)
        return pc.convert_continent_code_to_continent_name(continent_code)
    except:
        territory_fallbacks = {
            "VA": "Europe", "PS": "Asia", "XK": "Europe", "TW": "Asia", "HK": "Asia",
            "MO": "Asia", "CW": "North America", "SX": "North America", "TL": "Asia", "SS": "Africa"
        }
        return territory_fallbacks.get(iso_code, None)

def resolve_country_and_continent(place_name, raw_code):
    iso_code = None
    parts = [p.strip() for p in place_name.split(",") if p.strip()]
    if parts:
        last_part = parts[-1].lower()
        iso_code = country_name_to_iso.get(last_part)
        
    if not iso_code and raw_code:
        iso_code = fips_to_iso_crosswalk.get(raw_code.upper())
        
    if not iso_code and raw_code and len(raw_code) == 2:
        try:
            if pycountry.countries.get(alpha_2=raw_code.upper()):
                iso_code = raw_code.upper()
        except:
            pass
            
    if not iso_code:
        return None, None, None
        
    country_object = pycountry.countries.get(alpha_2=iso_code)
    clean_country_name = country_object.name if country_object else place_name
    continent = get_continent_for_country(iso_code)
    
    return clean_country_name, iso_code, continent

# Parse locations for all articles
location_results = []
for raw_loc_string in df_clean_step["raw_locations"]:
    matched_countries = []
    matched_codes = []
    matched_continents = []
    
    if raw_loc_string:
        for entry in raw_loc_string.split(";"):
            parts = entry.split("#")
            if len(parts) >= 3 and parts[1].strip():
                name_part = parts[1].strip()
                code_part = parts[2].strip()
                
                c_name, iso_2, cont = resolve_country_and_continent(name_part, code_part)
                if c_name and iso_2 and (c_name not in matched_countries):
                    matched_countries.append(c_name)
                    matched_codes.append(iso_2)
                    if cont and (cont not in matched_continents):
                        matched_continents.append(cont)
                        
    location_results.append({
        "country": matched_countries[0] if matched_countries else None,
        "country_code": matched_codes[0] if matched_codes else None,
        "continent": matched_continents[0] if matched_continents else None,
        "all_countries": "; ".join(matched_countries) if matched_countries else None,
        "all_country_codes": "; ".join(matched_codes) if matched_codes else None,
        "all_continents": "; ".join(matched_continents) if matched_continents else None
    })

df_locations = pd.DataFrame(location_results, index=df_clean_step.index)
for col in df_locations.columns:
    df_clean_step[col] = df_locations[col]

before_filter_count = len(df_clean_step)

# ==============================================================================
# GEOGRAPHIC FILTER: Only keep articles that have an identified country
# ==============================================================================
df_clean_step = df_clean_step[df_clean_step["country"].notnull()].copy()
after_filter_count = len(df_clean_step)
filtered_out = before_filter_count - after_filter_count

print(f"Total articles before geographic filter: {before_filter_count:,}")
print(f"Articles without an identified country filtered out: {filtered_out:,}")
print(f"✅ Retained articles with verified country and continent: {after_filter_count:,}")

print("\n--- Final Article Count by Continent ---")
print(df_clean_step["continent"].value_counts())


Total articles before geographic filter: 18,970
Articles without an identified country filtered out: 3,575
✅ Retained articles with verified country and continent: 15,395

--- Final Article Count by Continent ---
continent
North America    5010
Europe           4438
Asia             4346
Africa            822
Oceania           607
South America     172
Name: count, dtype: int64


## Step 6: Smart Topic Categorization (Fixing False Positives)

### The Problem We Solved:
In simple text matching, simple substring checks can cause hilarious mistakes:
- The keyword `"WAR"` matched `"WB_1174_WAREHOUSING_AND_STORAGE"` (warehousing), falsely labeling sunscreen and warehouse stories under *Conflict and Security*!
- The keyword `"KILL"` matched `"SKILLED"`.
- Weather tags like `"CRISISLEX_O01_WEATHER"` were accidentally matching crisis/conflict categories instead of *Environment and Climate*.

### Our Improved Hybrid Classification Strategy:
1. **Headline Priority:** Journalists write headlines to clearly tell the reader what the story is about. If a headline mentions *sunscreen*, *hospital*, *cancer*, or *vaccine*, it gets immediately and accurately classified as **Health**!
2. **Word-Boundary Theme Matching:** In GDELT themes, we require exact word boundaries (e.g. `WAR` must be a standalone word or surrounded by underscores, never inside `WAREHOUSING`).
3. **The 11 Target Topic Categories:**
   - **Politics and Government**
   - **Business and Economy**
   - **Technology**
   - **Health**
   - **Environment and Climate**
   - **Science**
   - **Conflict and Security**
   - **Education**
   - **Society and Culture**
   - **Other** (unmatched themes)
   - **Unknown** (no themes recorded)


In [7]:
# ==============================================================================
# Step 6: Smart Hybrid Topic Categorization (Title + Word-Bounded Themes)
# ==============================================================================

# 1. High-signal regular expressions for article headlines
TITLE_PATTERNS = {
    "Health": r"\b(?:health|medical|disease|virus|hospital|vaccine|pharmaceutical|doctor|cancer|medicine|surgery|patient|sunscreen|spf|mental health|pandemic|epidemic|clinic|infection|dementia|alzheimer|diabetes|obesity|diet|nurse|wellness|burn(?:t|ing)?)\b",
    "Environment and Climate": r"\b(?:climate|global warming|pollution|renewable|flood|flooding|earthquake|natural disaster|drought|wildfire|storm|hurricane|typhoon|heatwave|tornado|ecology|wildlife|emissions|weather|forecast|rain|snow|carbon)\b",
    "Conflict and Security": r"\b(?:war|military|terror(?:ist|ism)?|kill(?:ed|ing)?|attack(?:ed|s)?|security|crime|weapon|arrest(?:ed)?|protest|siege|armed conflict|insurgency|police|murder|hostage|missile|bomb(?:ing)?|troops|soldier|gunfire|shooting|gun|assault)\b",
    "Politics and Government": r"\b(?:politics|political|election|vote|voters|president|prime minister|minister|parliament|congress|senate|democracy|corruption|diplomat|diplomacy|treaty|legislation|constitutional|republican|democrat|governor|mayor)\b",
    "Business and Economy": r"\b(?:economy|economic|business|trade|finance|financial|market|stock(?:s)?|inflation|bank(?:ing)?|investment|investor|commerce|employment|unemployment|labor|tariff|currency|debt|revenue|gdp|real estate|earnings|shares)\b",
    "Technology": r"\b(?:technology|tech|cyber|internet|software|artificial intelligence|\bai\b|innovation|telecom|mobile|computer|data|digital|algorithm|robot(?:ics)?|automation|app|smartphone|crypto|semiconductor|chip)\b",
    "Science": r"\b(?:science|scientific|research(?:ers)?|space|physics|biology|astronomy|discovery|genetics|archaeology|neuroscience|chemistry|nasa|telescope|planet|quantum)\b",
    "Education": r"\b(?:education|school|schools|university|universities|student|students|college|teacher|teachers|academic|literacy|curriculum|scholarship|tuition|campus)\b",
    "Society and Culture": r"\b(?:culture|cultural|religion|religious|arts|media|entertainment|sports|football|soccer|cricket|tennis|tourism|community|human rights|gender|immigration|refugees|society|music|movie|film|actor|actress|festival|museum)\b"
}

# 2. Strict word-bounded regular expressions for GDELT theme tokens
# The pattern (?:^|_)...(?:_|$) prevents 'WAR' from matching inside 'WAREHOUSING'
THEME_PATTERNS = {
    "Conflict and Security": r"(?:^|_)(?:WAR|MILITARY|TERROR|TERRORISM|KILL|KILLING|KILLED|ATTACK|ATTACKS|SECURITY|CRIME|WEAPON|WEAPONS|ARREST|ARRESTS|PROTEST|PROTESTS|SEIGE|BLOCKADE|ARMEDCONFLICT|INSURGENCY|POLICE|JUSTICE|COURT|EXTREMISM|VIOLENCE|DEFENSE|FIREARM|FIREARMS|MURDER)(?:_|$)",
    "Politics and Government": r"(?:^|_)(?:POLITICS|POLITICAL|ELECTION|ELECTIONS|LEADER|LEADERS|GOV|PRESIDENT|MINISTER|PRIME_MINISTER|LEGISLATION|CONSTITUTIONAL|DEMOCRACY|CORRUPTION|PARLIAMENT|SENATE|CONGRESS|DIPLOMACY|TREATY|REGULATION|SOVEREIGNTY)(?:_|$)",
    "Business and Economy": r"(?:^|_)(?:ECON|ECONOMIC|ECONOMY|BUSINESS|TRADE|FINANCE|FINANCIAL|MARKET|MARKETS|STOCK|STOCKS|INFLATION|BANK|BANKING|INVESTMENT|INVESTOR|COMMERCE|EMPLOYMENT|UNEMPLOYMENT|LABOR|TARIFF|TARIFFS|CURRENCY|DEBT|REVENUE|AGRICULTURE)(?:_|$)",
    "Technology": r"(?:^|_)(?:TECH|TECHNOLOGY|CYBER|INTERNET|SOFTWARE|ARTIFICIAL_INTELLIGENCE|INNOVATION|TELECOM|MOBILE|COMPUTER|COMPUTERS|DIGITAL|ALGORITHM|ROBOTICS|AUTOMATION)(?:_|$)",
    "Health": r"(?:^|_)(?:HEALTH|MEDICAL|DISEASE|DISEASES|PANDEMIC|EPIDEMIC|VIRUS|HOSPITAL|HOSPITALS|VACCINE|VACCINES|PHARMACEUTICAL|DOCTOR|DOCTORS|CANCER|MEDICINE|SURGERY|PATIENT|PATIENTS|MENTAL_HEALTH)(?:_|$)",
    "Environment and Climate": r"(?:^|_)(?:ENV|CLIMATE|GLOBAL_WARMING|POLLUTION|RENEWABLE|FLOOD|FLOODS|EARTHQUAKE|EARTHQUAKES|NATURAL_DISASTER|DROUGHT|FORESTS|WATER_SECURITY|ECOLOGY|WILDLIFE|EMISSIONS|WEATHER)(?:_|$)",
    "Science": r"(?:^|_)(?:SCIENCE|SCIENTIFIC|RESEARCH|SPACE|PHYSICS|BIOLOGY|ASTRONOMY|DISCOVERY|GENETICS|ARCHAEOLOGY|NEUROSCIENCE|CHEMISTRY)(?:_|$)",
    "Education": r"(?:^|_)(?:EDUCATION|SCHOOL|SCHOOLS|UNIVERSITY|UNIVERSITIES|STUDENT|STUDENTS|COLLEGE|TEACHER|TEACHERS|ACADEMIC|LITERACY|CURRICULUM|SCHOLARSHIP)(?:_|$)",
    "Society and Culture": r"(?:^|_)(?:CULTURE|CULTURAL|RELIGION|RELIGIOUS|ARTS|MEDIA|ENTERTAINMENT|SPORTS|TOURISM|COMMUNITY|HUMAN_RIGHTS|GENDER|IMMIGRATION|REFUGEES|SOCIETY)(?:_|$)"
}

def classify_article_smart(title_text, raw_themes_str):
    title_matches = []
    title_lower = str(title_text).lower() if pd.notnull(title_text) else ""
    
    # Check title patterns
    for category, pattern in TITLE_PATTERNS.items():
        if re.search(pattern, title_lower):
            title_matches.append(category)
            
    # Check theme patterns with word boundaries
    theme_matches = set()
    if pd.notnull(raw_themes_str) and str(raw_themes_str).strip():
        tokens = str(raw_themes_str).split(";")
        for category, pattern in THEME_PATTERNS.items():
            for tok in tokens:
                tok = tok.strip().upper()
                if not tok: continue
                # Weather crisis belongs to Environment and Climate
                if tok.startswith("CRISISLEX_O01_WEATHER") or tok.startswith("WEATHER"):
                    if category == "Environment and Climate":
                        theme_matches.add(category)
                        break
                    continue
                if re.search(pattern, tok):
                    theme_matches.add(category)
                    break
                    
    # Combine title matches and theme matches, giving priority to title
    all_categories = list(dict.fromkeys(title_matches + sorted(list(theme_matches))))
    
    if not all_categories:
        if pd.notnull(raw_themes_str) and str(raw_themes_str).strip():
            return ["Other"], "Other"
        else:
            return ["Unknown"], "Unknown"
            
    primary_category = title_matches[0] if title_matches else all_categories[0]
    return all_categories, primary_category

topic_results = []
for _, row in df_clean_step.iterrows():
    cats, prim = classify_article_smart(row["title"], row["raw_themes"])
    topic_results.append({
        "topic": "; ".join(cats),
        "primary_topic": prim,
        "original_themes": str(row["raw_themes"])[:250] if pd.notnull(row["raw_themes"]) else None
    })

df_topics = pd.DataFrame(topic_results, index=df_clean_step.index)
for col in df_topics.columns:
    df_clean_step[col] = df_topics[col]

print("✅ Smart topic categorization complete!")
print("\n--- Distribution by Primary Topic ---")
print(df_clean_step["primary_topic"].value_counts())


✅ Smart topic categorization complete!

--- Distribution by Primary Topic ---
primary_topic
Business and Economy       6229
Conflict and Security      3192
Unknown                    1073
Politics and Government     961
Environment and Climate     851
Society and Culture         626
Education                   625
Health                      605
Technology                  560
Other                       513
Science                     160
Name: count, dtype: int64


## Step 7: Cleaning Publisher Names

We clean publisher domain names to ensure consistent grouping:
1. Converting to lowercase.
2. Removing `www.`, `m.`, `amp.`, and `mobile.` prefixes.
3. Removing port numbers (e.g., `:80`).
4. Using the URL hostname as fallback when the publisher field is blank.
5. Preserving the exact canonical article URL in `article_url`.


In [8]:
# ==============================================================================
# Step 7: Clean publisher domain names
# ==============================================================================

def clean_publisher_name(raw_domain, article_url):
    domain = raw_domain.strip().lower() if raw_domain else ""
    if not domain and article_url:
        try:
            domain = urllib.parse.urlparse(article_url).netloc.lower()
        except:
            domain = ""
            
    # Strip protocol and prefixes
    domain = re.sub(r"^(?:https?:\/\/)?(?:www\.|m\.|amp\.|mobile\.)+", "", domain)
    # Strip port numbers
    domain = domain.split(":")[0].strip()
    return domain

df_clean_step["publisher"] = [
    clean_publisher_name(r["raw_publisher"], r["article_url"]) 
    for _, r in df_clean_step.iterrows()
]

print("✅ Publisher cleaning complete!")
print("\n--- Top 10 News Publishers ---")
print(df_clean_step["publisher"].value_counts().head(10))


✅ Publisher cleaning complete!

--- Top 10 News Publishers ---
publisher
iheart.com              263
indiatimes.com          254
aol.co.uk               173
manilatimes.net         159
aol.com                 153
themarketsdaily.com     138
prokerala.com           134
finanznachrichten.de    121
freepressjournal.in     118
tickerreport.com        117
Name: count, dtype: int64


## Step 8: Quality Checks & Dataset Validation

Now we run automated quality assertions to verify that our final dataset is 100% clean and ready:
- All dates fall within July 14, 2026 – October 13, 2026.
- All article IDs are non-null and unique.
- **Zero missing countries!** Every article has a verified country, country code, and continent.
- All continent labels belong exclusively to `{Africa, Asia, Europe, North America, South America, Oceania}`.
- All URLs are well-formed web links.


In [9]:
# ==============================================================================
# Step 8: Automated Quality Assertions
# ==============================================================================

# 1. Date Range Check
earliest_date = df_clean_step["publication_date"].min()
latest_date = df_clean_step["publication_date"].max()
assert earliest_date >= "2026-07-14", f"Date error: {earliest_date} is before July 14, 2026!"
assert latest_date <= "2026-10-13", f"Date error: {latest_date} is after October 13, 2026!"
print(f"✓ Date check passed: All articles fall between {earliest_date} and {latest_date}.")

# 2. Unique Article ID Check
assert df_clean_step["article_id"].isnull().sum() == 0, "Error: Null article IDs detected!"
assert df_clean_step["article_id"].nunique() == len(df_clean_step), "Error: Duplicate article IDs detected!"
print(f"✓ Identifier check passed: All {len(df_clean_step):,} article IDs are unique.")

# 3. Complete Country and Continent Verification (0 Missing Values!)
assert df_clean_step["country"].isnull().sum() == 0, "Error: Missing country detected!"
assert df_clean_step["country_code"].isnull().sum() == 0, "Error: Missing country_code detected!"
assert df_clean_step["continent"].isnull().sum() == 0, "Error: Missing continent detected!"
print("✓ Geographic completeness check passed: 100% of articles have a verified country and continent!")

# 4. Continent Label Check
valid_continent_set = {"Africa", "Asia", "Europe", "North America", "South America", "Oceania"}
observed_continents = set(df_clean_step["continent"].unique())
assert observed_continents.issubset(valid_continent_set), f"Error: Unexpected continent found: {observed_continents - valid_continent_set}"
print(f"✓ Continent check passed: All 6 standard inhabited continents are validly represented.")

# 5. URL Format Check
invalid_url_count = (~df_clean_step["article_url"].str.startswith("http")).sum()
assert invalid_url_count == 0, f"Error: {invalid_url_count} malformed URLs found!"
print("✓ URL check passed: All article URLs are valid web addresses.")

# Reorder columns to the standard required schema
FINAL_COLUMNS = [
    "article_id",
    "title",
    "publication_date",
    "country",
    "country_code",
    "continent",
    "topic",
    "publisher",
    "article_url",
    "all_countries",
    "all_country_codes",
    "all_continents",
    "primary_topic",
    "original_themes"
]

df_final = df_clean_step[FINAL_COLUMNS].copy()

print("\n" + "=" * 55)
print("             FINAL VALIDATION SUMMARY")
print("=" * 55)
print(f"  Total Cleaned & Geocoded Articles: {len(df_final):,}")
print(f"  Date Range Covered:                {earliest_date} to {latest_date}")
print(f"  Number of Unique News Outlets:     {df_final['publisher'].nunique():,}")
print(f"  Countries Covered:                 {df_final['country'].nunique():,} countries")
print(f"  Continents Represented:            {len(observed_continents)} / 6 continents")
print(f"  Articles with Complete Geography:  100.0% ({len(df_final):,} / {len(df_final):,})")
print("=" * 55)


✓ Date check passed: All articles fall between 2026-07-14 and 2026-10-13.
✓ Identifier check passed: All 15,395 article IDs are unique.
✓ Geographic completeness check passed: 100% of articles have a verified country and continent!
✓ Continent check passed: All 6 standard inhabited continents are validly represented.
✓ URL check passed: All article URLs are valid web addresses.

             FINAL VALIDATION SUMMARY
  Total Cleaned & Geocoded Articles: 15,395
  Date Range Covered:                2026-07-14 to 2026-10-13
  Number of Unique News Outlets:     2,117
  Countries Covered:                 185 countries
  Continents Represented:            6 / 6 continents
  Articles with Complete Geography:  100.0% (15,395 / 15,395)


## Step 9: Saving the Final Deliverables

We export the two required tabular deliverables:
1. **`gdelt_clean.csv`**: The cleaned, fully geocoded worldwide news dataset.
2. **`data_dictionary.csv`**: Complete metadata documenting each column, data types, descriptions, and handling rules.


In [10]:
# ==============================================================================
# Step 9: Save gdelt_clean.csv and data_dictionary.csv
# ==============================================================================

current_directory = os.getcwd()
clean_dataset_path = os.path.join(current_directory, "gdelt_clean.csv")
data_dictionary_path = os.path.join(current_directory, "data_dictionary.csv")

# 1. Export the cleaned news dataset
df_final.to_csv(clean_dataset_path, index=False, encoding="utf-8")
print(f"✓ Saved gdelt_clean.csv: {os.path.getsize(clean_dataset_path):,} bytes ({len(df_final):,} rows)")

# 2. Define data dictionary entries
dictionary_rows = [
    {
        "column_name": "article_id",
        "data_type": "string",
        "description": "Unique, stable identifier for the news article record.",
        "source_or_derivation": "Directly extracted from GDELT 2.0 Global Knowledge Graph GKGRECORDID (format YYYYMMDDHHMMSS-index).",
        "missing_value_handling": "Mandatory primary key. Records without an identifier are excluded during initial extraction."
    },
    {
        "column_name": "title",
        "data_type": "string",
        "description": "Headline or page title of the published news article.",
        "source_or_derivation": "Parsed from <PAGE_TITLE> tag in GDELT ExtrasXML, unescaped for HTML entities and stripped of excess whitespace.",
        "missing_value_handling": "Preserved as empty string/null if omitted by the crawling source. Not artificially imputed."
    },
    {
        "column_name": "publication_date",
        "data_type": "string (YYYY-MM-DD)",
        "description": "Publication date of the article in ISO 8601 calendar format.",
        "source_or_derivation": "Converted from GDELT V2.1DATE timestamp (YYYYMMDDHHMMSS) to YYYY-MM-DD.",
        "missing_value_handling": "Mandatory field. Strictly filtered to fall between 2026-07-14 and 2026-10-13 inclusive."
    },
    {
        "column_name": "country",
        "data_type": "string",
        "description": "Standardized country name of the primary geographic location covered in the article text.",
        "source_or_derivation": "Derived from the primary location entity in GDELT V1Locations, resolved to ISO 3166-1 country names via pycountry.",
        "missing_value_handling": "Mandatory for mapping. Articles lacking an identifiable country are excluded so 100% of rows have verified geography."
    },
    {
        "column_name": "country_code",
        "data_type": "string (ISO 3166-1 alpha-2)",
        "description": "Standard two-letter ISO 3166-1 alpha-2 country code for the primary country covered in the article.",
        "source_or_derivation": "Crosswalked from GDELT location mentions and FIPS 10-4 country codes to ISO 3166-1 alpha-2.",
        "missing_value_handling": "Mandatory field. Fully populated for all retained articles."
    },
    {
        "column_name": "continent",
        "data_type": "string",
        "description": "Standard geographic continent name corresponding to the primary country covered in the article.",
        "source_or_derivation": "Mapped from ISO 3166-1 alpha-2 country code using pycountry-convert and UN M49 Geoscheme divisions.",
        "missing_value_handling": "Mandatory field. Fully populated across the 6 standard inhabited continents."
    },
    {
        "column_name": "topic",
        "data_type": "string",
        "description": "Standardized topic category (or semicolon-separated list of categories) reflecting article subject matter.",
        "source_or_derivation": "Mapped using hybrid headline analysis and word-bounded GDELT V1Themes tokens into the 11 project categories.",
        "missing_value_handling": "Classified as 'Unknown' if no themes are recorded by GDELT, or 'Other' if themes do not match standard categories."
    },
    {
        "column_name": "publisher",
        "data_type": "string",
        "description": "Standardized news publisher or media domain name.",
        "source_or_derivation": "Derived from GDELT V2SOURCECOMMONNAME or article URL host; lowercase with www., m., and amp. prefixes removed.",
        "missing_value_handling": "Fallback to URL hostname if source common name is blank. Never null."
    },
    {
        "column_name": "article_url",
        "data_type": "string (URL)",
        "description": "Full canonical URL of the original online news article.",
        "source_or_derivation": "Extracted directly from GDELT V2DOCUMENTIDENTIFIER with original parameters preserved.",
        "missing_value_handling": "Mandatory field. Records with missing or malformed URLs are removed."
    },
    {
        "column_name": "all_countries",
        "data_type": "string",
        "description": "Semicolon-delimited list of all distinct standardized countries mentioned in the article.",
        "source_or_derivation": "Aggregated from all valid locations parsed from GDELT V1Locations without row duplication.",
        "missing_value_handling": "Preserved as primary country if only one country is mentioned."
    },
    {
        "column_name": "all_country_codes",
        "data_type": "string",
        "description": "Semicolon-delimited list of all distinct ISO 3166-1 alpha-2 country codes mentioned in the article.",
        "source_or_derivation": "Mapped from all valid locations parsed from GDELT V1Locations.",
        "missing_value_handling": "Preserved as primary country code if only one country is mentioned."
    },
    {
        "column_name": "all_continents",
        "data_type": "string",
        "description": "Semicolon-delimited list of all distinct continents associated with the countries mentioned in the article.",
        "source_or_derivation": "Mapped from all_country_codes via continental reference.",
        "missing_value_handling": "Preserved as primary continent if only one continent is mentioned."
    },
    {
        "column_name": "primary_topic",
        "data_type": "string",
        "description": "Primary/first matched topic category among project categories.",
        "source_or_derivation": "Highest priority category derived from headline patterns and theme tokens.",
        "missing_value_handling": "Classified as 'Unknown' if no themes available, 'Other' if unclassified."
    },
    {
        "column_name": "original_themes",
        "data_type": "string",
        "description": "Raw GDELT theme tokens extracted from the knowledge graph, preserving original source taxonomy.",
        "source_or_derivation": "Extracted directly from GDELT V1Themes.",
        "missing_value_handling": "Preserved as null/empty string when no themes were assigned by GDELT."
    }
]

df_data_dict = pd.DataFrame(dictionary_rows)
df_data_dict.to_csv(data_dictionary_path, index=False, encoding="utf-8")
print(f"✓ Saved data_dictionary.csv: {os.path.getsize(data_dictionary_path):,} bytes ({len(df_data_dict):,} rows)")

# Quick sanity check: Read both files back to prove they open cleanly
check_clean = pd.read_csv(clean_dataset_path, keep_default_na=False)
check_dict = pd.read_csv(data_dictionary_path)
assert len(check_clean) == len(df_final), "Count mismatch in clean dataset!"
assert len(check_dict) == len(df_data_dict), "Count mismatch in data dictionary!"
print("\n🎉 Success! Both CSV files were re-read and verified 100% intact.")


✓ Saved gdelt_clean.csv: 9,194,120 bytes (15,395 rows)
✓ Saved data_dictionary.csv: 3,872 bytes (14 rows)



🎉 Success! Both CSV files were re-read and verified 100% intact.


## Step 10: Methodology Summary & Limitations (Student Quick Reference)

### 1. Data Source & Dates
- **Source:** GDELT 2.0 Global Knowledge Graph (GKG) open data stream.
- **Dates Covered:** July 14, 2026 to October 13, 2026 (92 days, exactly 3 months).
- **Scope:** Worldwide news coverage spanning all 6 inhabited continents.

### 2. Record Yield Summary
- **Raw records retrieved:** 19,406 records across 19 timestamps.
- **Syndicated near-duplicates removed:** 436 duplicate records.
- **Non-geographic articles filtered out:** 3,575 articles (articles lacking any country were excluded so that our geographic mapping team has 100% geolocated data).
- **Final clean, geocoded records:** **15,395 articles**.

### 3. Cleaning & Standardization Rules
- **Content Country vs. Publisher Country:** We always mapped the country *the story is about* (`V1Locations`), not the country where the website server is located.
- **International Standards:** Official ISO 3166-1 alpha-2 two-letter country codes (e.g. `US`, `GB`, `FR`, `NG`, `JP`, `AU`) and UN M49 continental groupings.
- **Smart Topic Categorization:** Combined headline keyword analysis with word-bounded theme tokens to prevent false-positive matches (such as warehousing matching war).
- **Publisher Names:** Cleaned by lowercasing, stripping `www.`, `m.`, `amp.`, and removing port numbers.

### 4. Known Biases and Limitations
Keep these real-world data science nuances in mind when analyzing the dataset:
1. **Language & Western Bias:** Even though GDELT translates over 65 languages, English-language media and Western news syndicates publish at much higher volumes than media in smaller nations.
2. **Automated Geocoding:** GDELT automatically detects place names in text. Some place names might refer to public buildings or metonyms (e.g. "the White House").
3. **Syndication Density:** A single breaking news event is often republished by multiple regional papers with identical or slightly modified headlines.

---
**Handoff Ready:** The files `gdelt_clean.csv`, `data_dictionary.csv`, and `data_preparation.ipynb` are completely ready.
